# Stage 6: State-Level Cybercrime Profile Clustering
**Project**: Cyber Crime Analytics for National Security  
**Data Source**: National Crime Records Bureau (NCRB) 2023 Master Dataset (`master_state_2023.csv`)  
**Methodological Position**: Unsupervised Profile Clustering on Standardized Composition Indicators ($N = 36$)

---

## 1. Objective & Scope

### Methodological Framing:
The objective of this stage is to identify **descriptive State/UT cybercrime profile groups** in the validated 2023 NCRB dataset using **K-Means clustering**.

### Critical Methodological Guardrails:
1. **Scale vs. Composition**: Clustering directly on raw case counts causes total crime volume / state population scale to dominate the Euclidean distance metric. To capture genuine *crime typology and motive composition*, clustering is performed on **non-redundant proportions and motive share indicators**.
2. **Descriptive Profile Groups**: Clusters describe groups of States/UTs with similar composition profiles in the observed 2023 cross-sectional dataset. They **do not** imply causality, geographic determinism, homogeneous intra-state behavior, or value judgments (e.g., no "high-risk" or "criminal" labels).
3. **Sample Size ($N = 36$)**: With 36 aggregate State/UT observations, clustering is evaluated across $K = 2$ through $8$ to balance silhouette cohesion and cluster granularity.


In [1]:
# Setup environment and imports
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 6 clustering module
from src.clustering import (
    load_and_prepare_features,
    evaluate_k_range,
    run_sensitivity_diagnostics,
    fit_final_kmeans,
    generate_cluster_assignments,
    generate_cluster_profiles,
    plot_clustering_elbow,
    plot_clustering_silhouette,
    plot_cluster_sizes,
    plot_cluster_feature_profiles,
    plot_cluster_projection,
    export_clustering_outputs,
    CLUSTERING_FEATURES,
    FEATURE_DESCRIPTIONS,
    CLUSTER_DESCRIPTIONS_K4
)

print(f"Working Directory: {project_root}")
print("Clustering module loaded successfully.")


Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics
Clustering module loaded successfully.


---
## Section B — Feature Selection & Distribution Inspection

### Feature Selection Rationale:
We select 4 non-overlapping, composition-based indicators that represent distinct legal and motivational dimensions:
1. **`it_act_share`**: Proportion of cases registered under the Information Technology Act vs. IPC (Legal framework composition).
2. **`fraud_motive_share`**: Proportion of cases driven by financial fraud motive (Dominant economic crime dimension).
3. **`extortion_motive_share`**: Proportion of cases driven by extortion / coercive threat motives (Violent/coercive dimension).
4. **`sexual_exploitation_motive_share`**: Proportion of cases driven by sexual exploitation / harassment motives (Interpersonal/morals dimension).


In [2]:
# Load master dataset and prepare clustering features
data_path = project_root / 'data' / 'processed' / 'master_state_2023.csv'
raw_df, feature_summary, X_scaled, scaler, feature_cols = load_and_prepare_features(data_path)

print(f"Prepared clustering dataset with shape: {raw_df.shape} (36 States/UTs)")
print("\n--- Candidate Feature Distribution Summary ---")
display(feature_summary[['feature_name', 'min', 'mean', 'median', 'max', 'std', 'skewness', 'description']])


Prepared clustering dataset with shape: (36, 11) (36 States/UTs)

--- Candidate Feature Distribution Summary ---


,feature_name,min,mean,median,max,std,skewness,description
0,it_act_share,0.0,0.6320,0.6875,1.0000,0.3386,-0.4421,Share of cases under IT Act relative to total state cybercrimes
1,fraud_motive_share,0.0,0.4885,0.4923,0.9518,0.2780,-0.2699,Share of financial fraud motive relative to state motive total
2,extortion_motive_share,0.0,0.0424,0.0242,0.1791,0.0482,1.3496,Share of extortion motive relative to state motive total
3,sexual_exploitation_motive_share,0.0,0.1600,0.1026,1.0000,0.2151,2.6717,Share of sexual exploitation motive relative to state motive total


In [3]:
# Feature Correlation Matrix
corr_matrix = raw_df[feature_cols].corr()
print("--- Feature Correlation Matrix ---")
display(corr_matrix.round(3))


--- Feature Correlation Matrix ---


,it_act_share,fraud_motive_share,extortion_motive_share,sexual_exploitation_motive_share
it_act_share,1.000,-0.029,0.008,0.343
fraud_motive_share,-0.029,1.000,-0.203,-0.535
extortion_motive_share,0.008,-0.203,1.000,-0.066
sexual_exploitation_motive_share,0.343,-0.535,-0.066,1.000


---
## Section C — Candidate K Evaluation (Elbow & Silhouette Analysis)

To identify a reasonable number of clusters, we evaluate $K \in [2, 8]$ using:
- **Inertia (Within-Cluster Sum of Squares)**: Measures cluster compactness.
- **Silhouette Coefficient**: Measures how well-separated and cohesive clusters are.
- **Cluster Size Distribution**: Monitors cluster fragmentation and tiny groups ($n \le 2$).


In [4]:
# Evaluate K from 2 to 8
eval_df = evaluate_k_range(X_scaled, k_range=range(2, 9), random_state=42)

print("--- K-Means Clustering Evaluation Table ---")
display(eval_df[['K', 'inertia', 'silhouette_score', 'min_cluster_size', 'max_cluster_size', 'clusters_n_le_2', 'cluster_sizes']])


--- K-Means Clustering Evaluation Table ---


,K,inertia,silhouette_score,min_cluster_size,max_cluster_size,clusters_n_le_2,cluster_sizes
0,2,106.9685,0.2557,13,23,0,"{0: 13, 1: 23}"
1,3,76.3459,0.2635,2,18,1,"{2: 2, 1: 16, 0: 18}"
2,4,49.6849,0.3497,2,15,1,"{2: 2, 3: 7, 1: 12, 0: 15}"
3,5,37.4862,0.3632,2,11,1,"{1: 2, 3: 7, 0: 7, 4: 9, 2: 11}"
4,6,30.4373,0.3449,2,8,1,"{2: 2, 0: 5, 4: 7, 1: 7, 3: 7, 5: 8}"
5,7,24.8073,0.3688,2,8,2,"{2: 2, 3: 2, 0: 5, 1: 6, 4: 6, 5: 7, 6: 8}"
6,8,20.0837,0.3771,2,8,2,"{6: 2, 2: 2, 5: 3, 4: 4, 3: 5, 7: 6, 1: 6, 0: 8}"


In [5]:
# Plot Elbow Method (K vs. Inertia)
fig_elbow = plot_clustering_elbow(
    eval_df,
    selected_k=4,
    save_path=str(project_root / 'outputs' / 'figures' / '15_clustering_elbow.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [6]:
# Plot Silhouette Analysis (K vs. Silhouette Score)
fig_silhouette = plot_clustering_silhouette(
    eval_df,
    selected_k=4,
    save_path=str(project_root / 'outputs' / 'figures' / '16_clustering_silhouette.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section D — Selected K Rationale & Sensitivity Diagnostics

### Selection Justification for $K = 4$:
- **Elbow Inflection**: Inertia drops by $34.9\%$ from $K=3$ to $K=4$ ($76.35 \to 49.68$), after which marginal inertia reduction slows.
- **Silhouette Coefficient Step-Up**: The silhouette coefficient steps up significantly from $0.2635$ ($K=3$) to **$0.3497$ ($K=4$)** ($+32.7\%$ gain).
- **Interpretable Compromise**: While $K=5$, $7$, and $8$ yield marginally higher silhouette scores ($0.3632$, $0.3688$, $0.3771$), they introduce additional micro-clusters ($n \le 2$) and over-fragment the small sample of $36$ jurisdictions without adding distinct profile interpretations. $K=4$ was selected as an interpretable compromise between cluster separation, elbow structure, and cluster fragmentation.


In [7]:
# Diagnostic 1: Two-State Small-Denominator Cluster Inspection
print("--- Two-State Cluster Diagnostic (Cluster 2) ---")
two_states = raw_df[raw_df['state_name'].isin(['Dadra and Nagar Haveli and Daman and Diu', 'Lakshadweep'])]
display(two_states[['state_name', 'total_cases', 'it_act_cases', 'motive_total', 'motive_fraud',
                    'motive_sexual_exploitation', 'it_act_share', 'sexual_exploitation_motive_share']])


--- Two-State Cluster Diagnostic (Cluster 2) ---


,state_name,total_cases,it_act_cases,motive_total,motive_fraud,motive_sexual_exploitation,it_act_share,sexual_exploitation_motive_share
30,Dadra and Nagar Haveli and Daman and Diu,6,6,6,0,5,1.0,0.833333
34,Lakshadweep,1,1,1,0,1,1.0,1.000000


In [8]:
# Diagnostic 2: Sensitivity Analysis & Hungarian Membership Stability Audit
sens_df, stability_df = run_sensitivity_diagnostics(raw_df, X_scaled, feature_cols)
print("--- Clustering Sensitivity Summary Table ---")
display(sens_df[['model_name', 'sample_size', 'features_count', 'K', 'inertia', 'silhouette_score', 'cluster_sizes', 'agreement_vs_primary']])

print("\n--- State-by-State Membership Stability Table (Hungarian Label Alignment) ---")
display(stability_df)


--- Clustering Sensitivity Summary Table ---

--- State-by-State Membership Stability Table (Hungarian Label Alignment) ---


,model_name,sample_size,features_count,K,inertia,silhouette_score,cluster_sizes,agreement_vs_primary
0,"Primary Model (N=36, 4 Features)",36,4,4,49.6849,0.3497,"{0: 15, 1: 12, 3: 7, 2: 2}",100.0% (36/36)
1,"Sample Exclusion Sensitivity (N=34, 4 Features)",34,4,4,52.0066,0.3281,"{2: 11, 1: 9, 0: 8, 3: 6}",76.5% (26/34)
2,"Feature Ablation Sensitivity (N=36, 3 Features)",36,3,4,36.3694,0.3523,"{1: 11, 2: 10, 0: 8, 3: 7}",75.0% (27/36)


,state_name,primary_k4_id,primary_k4_label,ablation_k4_aligned_id,ablation_k4_label,ablation_changed,n34_aligned_id,n34_changed
0,Andhra Pradesh,0,Lower IT Act Share / Moderate Fraud Share Profile,0,Lower IT Act Share / Moderate Fraud Share Profile,No,0.0,No
1,Arunachal Pradesh,1,Higher IT Act Share / Higher Fraud Share Profile,1,Higher IT Act Share / Higher Fraud Share Profile,No,1.0,No
2,Assam,3,Higher Extortion Motive Share Profile,3,Higher Extortion Motive Share Profile,No,3.0,No
3,Bihar,0,Lower IT Act Share / Moderate Fraud Share Profile,0,Lower IT Act Share / Moderate Fraud Share Profile,No,0.0,No
4,Chhattisgarh,0,Lower IT Act Share / Moderate Fraud Share Profile,2,High Sexual-Exploitation Share / Small-Denominator Profile,Yes,2.0,Yes
5,Goa,1,Higher IT Act Share / Higher Fraud Share Profile,1,Higher IT Act Share / Higher Fraud Share Profile,No,1.0,No
6,Gujarat,0,Lower IT Act Share / Moderate Fraud Share Profile,0,Lower IT Act Share / Moderate Fraud Share Profile,No,0.0,No
7,Haryana,0,Lower IT Act Share / Moderate Fraud Share Profile,2,High Sexual-Exploitation Share / Small-Denominator Profile,Yes,0.0,No
8,Himachal Pradesh,1,Higher IT Act Share / Higher Fraud Share Profile,1,Higher IT Act Share / Higher Fraud Share Profile,No,1.0,No
9,Jharkhand,1,Higher IT Act Share / Higher Fraud Share Profile,1,Higher IT Act Share / Higher Fraud Share Profile,No,1.0,No


In [9]:
# Membership Stability Summary Metrics
ablation_unchanged = (stability_df['ablation_changed'] == 'No').sum()
ablation_changed = (stability_df['ablation_changed'] == 'Yes').sum()
ablation_pct = (ablation_unchanged / 36) * 100

n34_sub = stability_df[stability_df['n34_changed'] != 'Excluded']
n34_unchanged = (n34_sub['n34_changed'] == 'No').sum()
n34_changed = (n34_sub['n34_changed'] == 'Yes').sum()
n34_pct = (n34_unchanged / 34) * 100

print(f"Feature Ablation (3-Feature K=4 vs Primary 4-Feature K=4):")
print(f"  - Unchanged: {ablation_unchanged} / 36 ({ablation_pct:.1f}%)")
print(f"  - Changed:   {ablation_changed} / 36 ({100 - ablation_pct:.1f}%)")
reassigned_ablation = stability_df[stability_df['ablation_changed'] == 'Yes']['state_name'].tolist()
print(f"  - Reassigned States/UTs: {', '.join(reassigned_ablation)}")

print(f"\nSample Exclusion (N=34 vs Primary N=36):")
print(f"  - Unchanged: {n34_unchanged} / 34 ({n34_pct:.1f}%)")
print(f"  - Changed:   {n34_changed} / 34 ({100 - n34_pct:.1f}%)")
reassigned_n34 = n34_sub[n34_sub['n34_changed'] == 'Yes']['state_name'].tolist()
print(f"  - Reassigned States/UTs: {', '.join(reassigned_n34)}")


Feature Ablation (3-Feature K=4 vs Primary 4-Feature K=4):
  - Unchanged: 27 / 36 (75.0%)
  - Changed:   9 / 36 (25.0%)
  - Reassigned States/UTs: Chhattisgarh, Haryana, Manipur, Rajasthan, Tamil Nadu, West Bengal, Andaman and Nicobar Islands, Delhi, Ladakh

Sample Exclusion (N=34 vs Primary N=36):
  - Unchanged: 26 / 34 (76.5%)
  - Changed:   8 / 34 (23.5%)
  - Reassigned States/UTs: Chhattisgarh, Manipur, Meghalaya, Rajasthan, Tripura, Andaman and Nicobar Islands, Chandigarh, Jammu and Kashmir


---
## Section E — Final Cluster Model & Profiles ($K = 4$)


In [10]:
# Fit final K-Means model with K = 4
selected_k = 4
km_model, labels = fit_final_kmeans(X_scaled, n_clusters=selected_k, random_state=42)

# Generate assignments and profiles
assignments_df = generate_cluster_assignments(
    raw_df, X_scaled, labels, feature_cols, cluster_names=CLUSTER_DESCRIPTIONS_K4
)
profiles_df = generate_cluster_profiles(
    raw_df, labels, feature_cols, cluster_names=CLUSTER_DESCRIPTIONS_K4
)

print("--- Final Cluster Profiles (K = 4) ---")
display(profiles_df[['cluster_id', 'cluster_label', 'state_count', 'state_pct',
                     'it_act_share_mean', 'fraud_motive_share_mean',
                     'extortion_motive_share_mean', 'sexual_exploitation_motive_share_mean']])


--- Final Cluster Profiles (K = 4) ---


,cluster_id,cluster_label,state_count,state_pct,it_act_share_mean,fraud_motive_share_mean,extortion_motive_share_mean,sexual_exploitation_motive_share_mean
0,0,Lower IT Act Share / Moderate Fraud Share Profile,15,41.67,0.3267,0.4442,0.0276,0.1103
1,1,Higher IT Act Share / Higher Fraud Share Profile,12,33.33,0.8868,0.7161,0.0194,0.1053
2,2,High Sexual-Exploitation Share / Small-Denominator Profile,2,5.56,1.0000,0.0000,0.0000,0.9167
3,3,Higher Extortion Motive Share Profile,7,19.44,0.7440,0.3325,0.1254,0.1438


In [11]:
# Cluster Membership Breakdown (State/UT Members)
print("--- Cluster Membership Breakdown (State/UT Members) ---")
for cid in range(selected_k):
    sub = assignments_df[assignments_df['cluster_id'] == cid]
    c_name = CLUSTER_DESCRIPTIONS_K4[cid]
    print(f"\nCluster {cid}: {c_name} (n = {len(sub)}, {len(sub)/36*100:.1f}%):")
    print("  State/UT Members: " + ", ".join(sub['state_name'].tolist()))


--- Cluster Membership Breakdown (State/UT Members) ---

Cluster 0: Lower IT Act Share / Moderate Fraud Share Profile (n = 15, 41.7%):
  State/UT Members: Andaman and Nicobar Islands, Andhra Pradesh, Bihar, Chhattisgarh, Delhi, Gujarat, Haryana, Ladakh, Madhya Pradesh, Maharashtra, Manipur, Odisha, Rajasthan, Telangana, West Bengal

Cluster 1: Higher IT Act Share / Higher Fraud Share Profile (n = 12, 33.3%):
  State/UT Members: Arunachal Pradesh, Goa, Himachal Pradesh, Jammu and Kashmir, Jharkhand, Karnataka, Meghalaya, Mizoram, Nagaland, Puducherry, Tamil Nadu, Tripura

Cluster 2: High Sexual-Exploitation Share / Small-Denominator Profile (n = 2, 5.6%):
  State/UT Members: Dadra and Nagar Haveli and Daman and Diu, Lakshadweep

Cluster 3: Higher Extortion Motive Share Profile (n = 7, 19.4%):
  State/UT Members: Assam, Chandigarh, Kerala, Punjab, Sikkim, Uttar Pradesh, Uttarakhand


---
## Section F — Cluster Visualizations & Projections


In [12]:
# Plot Cluster Sizes
fig_sizes = plot_cluster_sizes(
    assignments_df,
    cluster_names=CLUSTER_DESCRIPTIONS_K4,
    save_path=str(project_root / 'outputs' / 'figures' / '17_cluster_sizes.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [13]:
# Plot Comparative Feature Means Across Clusters
fig_profiles = plot_cluster_feature_profiles(
    profiles_df,
    feature_cols,
    save_path=str(project_root / 'outputs' / 'figures' / '18_cluster_feature_profiles.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [14]:
# Plot 2D PCA Projection of Cluster Space (Visualization Aid)
fig_proj = plot_cluster_projection(
    X_scaled,
    assignments_df,
    cluster_names=CLUSTER_DESCRIPTIONS_K4,
    save_path=str(project_root / 'outputs' / 'figures' / '19_cluster_projection.png')
)
plt.show()


<string>:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section G — Descriptive Profile Interpretation

### Cluster 0 ($n = 15$ States/UTs, $41.67\%$): *Lower IT Act Share / Moderate Fraud Share Profile*
- **Key Feature Characteristics**: Lowest mean IT Act share ($32.67\%$, indicating predominance of IPC registrations such as Sec. 420 cheating), moderate mean Fraud motive share ($44.42\%$), low mean Extortion motive share ($2.76\%$), and moderate Sexual Exploitation motive share ($11.03\%$).
- **State/UT Members**: Maharashtra, Telangana, Bihar, Andhra Pradesh, Gujarat, Madhya Pradesh, Rajasthan, Delhi, West Bengal, Odisha, Chhattisgarh, Haryana, Manipur, Ladakh, Andaman and Nicobar Islands.

---

### Cluster 1 ($n = 12$ States/UTs, $33.33\%$): *Higher IT Act Share / Higher Fraud Share Profile*
- **Key Feature Characteristics**: Highest mean IT Act share ($88.68\%$) and highest mean Fraud motive share ($71.61\%$), with low mean Extortion motive share ($1.94\%$) and moderate Sexual Exploitation motive share ($10.53\%$).
- **State/UT Members**: Karnataka, Tamil Nadu, Jharkhand, Goa, Himachal Pradesh, Arunachal Pradesh, Mizoram, Nagaland, Meghalaya, Tripura, Jammu and Kashmir, Puducherry.

---

### Cluster 2 ($n = 2$ States/UTs, $5.56\%$): *High Sexual-Exploitation Share / Small-Denominator Profile*
- **Key Feature Characteristics**: Highest mean Sexual Exploitation motive share ($91.67\%$) and $100.00\%$ IT Act share, with $0.00\%$ Fraud motive share.
- **State/UT Members**: Dadra and Nagar Haveli and Daman and Diu ($6$ total reported cases), Lakshadweep ($1$ total reported case).
- **Substantive Caution**: This cluster reflects high proportional concentration in jurisdictions with very small total case counts ($N=6$ and $N=1$), **not high crime volume**.

---

### Cluster 3 ($n = 7$ States/UTs, $19.44\%$): *Higher Extortion Motive Share Profile*
- **Key Feature Characteristics**: Distinctly elevated mean Extortion motive share ($12.54\%$, approximately $3\times$ the national state average of $4.24\%$), moderate-to-high mean IT Act share ($74.40\%$), moderate Fraud motive share ($33.25\%$), and moderate Sexual Exploitation motive share ($14.38\%$).
- **State/UT Members**: Uttar Pradesh, Assam, Punjab, Kerala, Uttarakhand, Sikkim, Chandigarh.


In [15]:
# Export Tables for Downstream Analysis and Power BI
exported_tables = export_clustering_outputs(
    eval_df=eval_df,
    assignments_df=assignments_df,
    profiles_df=profiles_df,
    sens_df=sens_df,
    stability_df=stability_df,
    output_dir=str(project_root / 'outputs' / 'tables')
)

print("Exported Clustering Output Tables:")
for k, v in exported_tables.items():
    p = Path(v)
    print(f"  - {k:<15}: {p.name} ({p.stat().st_size:,} bytes)")


Exported Clustering Output Tables:
  - evaluation     : clustering_evaluation.csv (534 bytes)
  - assignments    : cluster_assignments_2023.csv (7,501 bytes)
  - profiles       : cluster_profiles_2023.csv (921 bytes)
  - sensitivity    : clustering_sensitivity_analysis.csv (734 bytes)
  - stability      : clustering_membership_stability.csv (4,624 bytes)


---
## Section H — Methodological Limitations & Analytical Boundaries

### Explicit Constraints:
1. **Sample Size ($N = 36$)**: Observations represent aggregate State/UT jurisdictions for the single reporting year 2023. Small sample size limits the geometric complexity of cluster boundaries.
2. **Composition vs. Absolute Volume**: These clusters group states based on *relative legal and motivational composition*. States with vastly different absolute case totals (e.g., Karnataka vs. Mizoram) belong to the same cluster because their proportional profiles are similar.
3. **K-Means Geometric Assumptions**: K-Means assumes spherical clusters of approximately equal variance in standardized space.
4. **Descriptive, Non-Causal Nature**: Clusters represent statistical groupings in the observed cross-sectional dataset. They do not test causal mechanisms or explain why specific profiles emerge.
